# 🏋️ Smart Gym Assistant — Kaggle Execution Notebook

**Academic Context:** Computer Engineering — *Visual Tracking & Intelligent Applications (2026-2027)*  
**Authors:** Loua Ben Afia & Abir Rouag  
**Supervisor:** Dr. Rihab Laajili, Polytechnic Institute  

This notebook runs the complete **Smart Gym Assistant** computer vision pipeline on Kaggle GPU/CPU instances:
1. **Pose Estimation:** MediaPipe Pose (12 core landmarks)
2. **Temporal Tracking:** Coordinate EMA smoothing & occlusion carry-forward
3. **Biomechanical FSM:** Finite State Machines for Squats & Bicep Curls
4. **Video Generation:** Exports annotated MP4 video with real-time HUD overlays
5. **Interactive Analytics:** Plotly angle trajectories, quality radar charts, and CSV/JSON export
6. **Optional Streamlit Cloud Tunnel:** Run the full interactive web dashboard from Kaggle

## 1. Environment Setup & Dependencies

In [ ]:
# Install required pinned packages
!pip install -q mediapipe==0.10.14 "numpy<2.0.0" opencv-python-headless plotly pyyaml scipy

## 2. Verify Repository Setup & Run Unit Tests

In [ ]:
import os, sys
# If running inside cloned folder or Kaggle input, ensure root is in sys.path
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Run unit tests (geometry, tracker, squat FSM, curl FSM, feedback)
!pytest -q

## 3. Run Headless Biomechanical Pipeline on Sample Video

You can specify any video from `/kaggle/input/` or the provided sample in `data/videos/squat_sample.mp4`.

In [ ]:
from run_kaggle import process_video_headless

input_video = "data/videos/squat_sample.mp4"
output_video = "/kaggle/working/squat_annotated.mp4" if os.path.exists("/kaggle") else "data/sessions/squat_annotated.mp4"
output_json = "/kaggle/working/session.json" if os.path.exists("/kaggle") else "data/sessions/session.json"
output_csv = "/kaggle/working/reps.csv" if os.path.exists("/kaggle") else "data/sessions/reps.csv"

session_data = process_video_headless(
    video_path=input_video,
    exercise_name="Squats",
    side_override="auto",
    output_video_path=output_video,
    output_json_path=output_json,
    output_csv_path=output_csv,
)

## 4. Play Annotated Video with Skeletal HUD in Notebook

In [ ]:
from IPython.display import HTML
import base64

if os.path.exists(output_video):
    with open(output_video, "rb") as f:
        video_bytes = f.read()
    encoded_video = base64.b64encode(video_bytes).decode("ascii")
    display(HTML(f'''
        <div style="text-align: center;">
            <h3>Annotated Workout Video with Live Biomechanical HUD</h3>
            <video width="640" height="360" controls autoplay muted loop>
                <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
                Your browser does not support the video tag.
            </video>
        </div>
    '''))
else:
    print("Output video file not found.")

## 5. Interactive Visualizations (Plotly Trajectory & Quality Radar)

In [ ]:
import pandas as pd
from app.components.charts import create_angle_trajectory_chart, create_radar_score_chart, create_gauge_chart
from core.models import FrameResult, RepEvent

# Load session result
reps = [RepEvent(**r) for r in session_data["reps"]]

# Reconstruct FrameResults from processed video
print(f"Overall Quality Score: {session_data['summary'].get('overall_form', 0)}%")
fig_radar = create_radar_score_chart(session_data["summary"])
fig_radar.show()

# Display per-rep table
if os.path.exists(output_csv):
    df_reps = pd.read_csv(output_csv)
    display(df_reps)
else:
    df_reps = pd.DataFrame(session_data["reps"])
    display(df_reps)

## 6. Run Benchmark Evaluation Suite on Kaggle

Executes `evaluate.py` across the test video dataset and generates summary tables grouped by lighting, angle, and occlusion.

In [ ]:
!python -m evaluation.evaluate --gt evaluation/ground_truth_template.csv --videos data/videos --out evaluation/report.csv

## 7. (Optional) Run Interactive Streamlit UI via Public Cloudflare Tunnel

To use the full multi-page Streamlit web application directly from Kaggle:

In [ ]:
# Launch Streamlit in the background and establish a secure public tunnel via cloudflared
import subprocess, time

print("Starting Streamlit...")
st_proc = subprocess.Popen(["streamlit", "run", "app/Home.py", "--server.port=8501", "--server.headless=true"])
time.sleep(3)

# Download cloudflared if not present
if not os.path.exists("cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

print("Launching tunnel... Click the URL printed below to open Streamlit:")
!./cloudflared tunnel --url http://localhost:8501